In [4]:
from pyvirtualdisplay import Display

virtual_display = Display(visible=0, size=(1400, 900))
virtual_display.start()

In [10]:
import gymnasium

from huggingface_sb3 import load_from_hub, package_to_hub
from huggingface_hub import (
    notebook_login,
)
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.monitor import Monitor

2026-03-24 14:51:18.206470: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774363878.365726      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774363878.411137      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774363878.785437      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774363878.785477      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774363878.785480      55 computation_placer.cc:177] computation placer alr

In [16]:
env = gymnasium.make('LunarLander-v3')

In [20]:
model = PPO(
    policy="MlpPolicy",
    env=env,
    n_steps=2048,
    batch_size=64,
    n_epochs=10,
    gamma=0.99,
    gae_lambda=0.95,
    learning_rate=3e-4,
    clip_range=0.2,
    ent_coef=0.01,
    vf_coef=0.5,
    max_grad_norm=0.5,
    verbose=1,
    device="cuda"
)

Using cuda device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


In [21]:
model.learn(total_timesteps=500000)

---------------------------------
| rollout/           |          |
|    ep_len_mean     | 92.3     |
|    ep_rew_mean     | -169     |
| time/              |          |
|    fps             | 697      |
|    iterations      | 1        |
|    time_elapsed    | 2        |
|    total_timesteps | 2048     |
---------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 93.2        |
|    ep_rew_mean          | -177        |
| time/                   |             |
|    fps                  | 568         |
|    iterations           | 2           |
|    time_elapsed         | 7           |
|    total_timesteps      | 4096        |
| train/                  |             |
|    approx_kl            | 0.004767682 |
|    clip_fraction        | 0.0227      |
|    clip_range           | 0.2         |
|    entropy_loss         | -1.38       |
|    explained_variance   | 0.00395     |
|    learning_rate        | 0.

In [23]:
eval_env = Monitor(gymnasium.make("LunarLander-v3"))
mean_reward, std_reward = evaluate_policy(model, eval_env, n_eval_episodes=10, deterministic=True)
print(f"mean_reward={mean_reward:.2f} +/- {std_reward}")

mean_reward=270.66 +/- 16.87588726582117


In [24]:
notebook_login()

In [32]:
from stable_baselines3.common.vec_env import DummyVecEnv, VecVideoRecorder
eval_env = DummyVecEnv([
    lambda: gymnasium.make(env_id, render_mode="rgb_array")
])
eval_env = VecVideoRecorder(
    eval_env,
    video_folder="./videos",
    record_video_trigger=lambda step: step == 0,
    video_length=1000,
    name_prefix="rl-agent"
)

In [33]:
env_id = "LunarLander-v3"
model_architecture = "PPO"
repo_id = 'ByteMeHarder-404/lunarlander-ppo1'
commit_message ='So long my friend'

In [34]:
package_to_hub(
    model=model,
    model_name='PPO',
    model_architecture=model_architecture,
    env_id=env_id,
    eval_env=eval_env,
    repo_id=repo_id,
    commit_message=commit_message,
)

ℹ This function will save, evaluate, generate a video of your agent,
create a model card and push everything to the hub. It might take up to 1min.
This is a work in progress: if you encounter a bug, please open an issue.


/usr/local/lib/python3.12/dist-packages/stable_baselines3/common/evaluation.py:70: UserWarning: Evaluation environment is not wrapped with a ``Monitor`` wrapper. This may result in reporting modified episode lengths and rewards, if other wrappers happen to modify these. Consider wrapping environment first with ``Monitor`` wrapper.
  warnings.warn(


Saving video to /kaggle/working/videos/rl-agent-step-0-to-step-1000.mp4
Moviepy - Building video /kaggle/working/videos/rl-agent-step-0-to-step-1000.mp4.
Moviepy - Writing video /kaggle/working/videos/rl-agent-step-0-to-step-1000.mp4



Moviepy - Done !
Moviepy - video ready /kaggle/working/videos/rl-agent-step-0-to-step-1000.mp4
Saving video to /tmp/tmp1_3jhzvb/-step-0-to-step-1000.mp4
Moviepy - Building video /tmp/tmp1_3jhzvb/-step-0-to-step-1000.mp4.
Moviepy - Writing video /tmp/tmp1_3jhzvb/-step-0-to-step-1000.mp4



Moviepy - Done !
Moviepy - video ready /tmp/tmp1_3jhzvb/-step-0-to-step-1000.mp4
✘ 'DummyVecEnv' object has no attribute 'video_recorder'
✘ We are unable to generate a replay of your agent, the package_to_hub
process continues
✘ Please open an issue at
https://github.com/huggingface/huggingface_sb3/issues
ℹ Pushing repo ByteMeHarder-404/lunarlander-ppo1 to the Hugging Face
Hub


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

ℹ Your model is pushed to the Hub. You can view your model here:
https://huggingface.co/ByteMeHarder-404/lunarlander-ppo1/tree/main/


CommitInfo(commit_url='https://huggingface.co/ByteMeHarder-404/lunarlander-ppo1/commit/fa512ac59e405012e1378e9c7a07f4473d543328', commit_message='So long my friend', commit_description='', oid='fa512ac59e405012e1378e9c7a07f4473d543328', pr_url=None, repo_url=RepoUrl('https://huggingface.co/ByteMeHarder-404/lunarlander-ppo1', endpoint='https://huggingface.co', repo_type='model', repo_id='ByteMeHarder-404/lunarlander-ppo1'), pr_revision=None, pr_num=None)